In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor

from combine_features import read_data

# Experiments

In [69]:
df = read_data("../rhea-soil-nutrient-prediction-challenge/Train.csv")
df.head()

,ID,Longitude,Latitude,Depth_cm,ph,Area,Cropland nitrogen per unit area,Cropland phosphorus per unit area,Cropland potassium per unit area,tmin_avg,...,Cu,Fe,Mg,Mn,N,P,K,Na,S,Zn
0,BF9XTB,37.65189,-3.15440,20-50,6.405,Kenya,14.202173,3.902236,-7.263155,15.240530,...,5.826,81.780,306.836,270.240,0.79,NaN,300.951,NaN,NaN,NaN
1,2RWYTR,37.63612,-3.08585,20-50,6.419,Kenya,14.202173,3.902236,-7.263155,15.240530,...,4.346,97.198,407.980,185.557,1.11,NaN,292.696,NaN,NaN,NaN
2,XZI9Q6,39.55580,-2.67218,20-50,8.388,Kenya,14.202173,3.902236,-7.263155,21.969696,...,3.657,42.672,1256.319,178.299,0.45,NaN,814.911,NaN,NaN,NaN
3,4CBCVY,39.55477,-2.67196,20-50,8.302,Kenya,14.202173,3.902236,-7.263155,21.969696,...,3.376,52.861,1322.732,464.137,0.31,NaN,815.337,NaN,NaN,NaN
4,F9GK9S,39.55477,-2.67196,20-50,8.292,Kenya,14.202173,3.902236,-7.263155,21.969696,...,3.351,46.057,1134.898,274.565,0.45,NaN,928.238,NaN,NaN,NaN


In [70]:
df.iloc[:, 40:].info()

<class 'pandas.DataFrame'>
RangeIndex: 13454 entries, 0 to 13453
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Ca         13454 non-null  float64
 1   C_organic  13454 non-null  float64
 2   C_total    13416 non-null  float64
 3   Cu         13416 non-null  float64
 4   Fe         13416 non-null  float64
 5   Mg         13454 non-null  float64
 6   Mn         13416 non-null  float64
 7   N          13416 non-null  float64
 8   P          0 non-null      float64
 9   K          13454 non-null  float64
 10  Na         38 non-null     float64
 11  S          0 non-null      float64
 12  Zn         0 non-null      float64
dtypes: float64(13)
memory usage: 1.3 MB


In [71]:
pred_to_keep = pd.read_csv("../rhea-soil-nutrient-prediction-challenge/TargetPred_To_Keep.csv")
pred_to_keep.head()

,ID,Al,B,Ca,Cu,Fe,K,Mg,Mn,N,Na,P,S,Zn
0,00A83S,1,1,1,1,1,1,1,1,1,1,1,1,1
1,00F2Q3,1,1,1,1,1,1,1,1,1,1,1,1,1
2,00FNFP,1,0,1,1,1,1,1,1,1,0,0,0,0
3,01MFSS,1,0,1,1,1,1,1,1,1,0,0,0,0
4,02851F,1,0,1,1,1,1,1,1,1,0,0,0,0


In [72]:
pred_to_keep.sum()

ID    00A83S00F2Q300FNFP01MFSS02851F02C3Q502I9O502L2...
Al                                                 6070
B                                                  1345
Ca                                                 6070
Cu                                                 6065
Fe                                                 6065
K                                                  6070
Mg                                                 6070
Mn                                                 6065
N                                                  6065
Na                                                 1350
P                                                  1345
S                                                  1345
Zn                                                 1345
dtype: object

In [73]:
encoder = LabelEncoder()
df['Depth'] = encoder.fit_transform(df['Depth_cm'])
# df.drop(columns=['ID', 'Area', 'Depth_cm'], inplace=True)
# df.dropna(axis=1, thresh=int(0.8 * df.shape[0]), inplace=True) # drops target columns :(
# df.dropna(axis=0, inplace=True)
df.head()

,ID,Longitude,Latitude,Depth_cm,ph,Area,Cropland nitrogen per unit area,Cropland phosphorus per unit area,Cropland potassium per unit area,tmin_avg,...,Fe,Mg,Mn,N,P,K,Na,S,Zn,Depth
0,BF9XTB,37.65189,-3.15440,20-50,6.405,Kenya,14.202173,3.902236,-7.263155,15.240530,...,81.780,306.836,270.240,0.79,NaN,300.951,NaN,NaN,NaN,1
1,2RWYTR,37.63612,-3.08585,20-50,6.419,Kenya,14.202173,3.902236,-7.263155,15.240530,...,97.198,407.980,185.557,1.11,NaN,292.696,NaN,NaN,NaN,1
2,XZI9Q6,39.55580,-2.67218,20-50,8.388,Kenya,14.202173,3.902236,-7.263155,21.969696,...,42.672,1256.319,178.299,0.45,NaN,814.911,NaN,NaN,NaN,1
3,4CBCVY,39.55477,-2.67196,20-50,8.302,Kenya,14.202173,3.902236,-7.263155,21.969696,...,52.861,1322.732,464.137,0.31,NaN,815.337,NaN,NaN,NaN,1
4,F9GK9S,39.55477,-2.67196,20-50,8.292,Kenya,14.202173,3.902236,-7.263155,21.969696,...,46.057,1134.898,274.565,0.45,NaN,928.238,NaN,NaN,NaN,1


In [74]:
target = [
        "Al",
        "B",
        "Ca",
        "C_organic",
        "C_total",
        "Cu",
        "Fe",
        "K",
        "Mg",
        "Mn",
        "N",
        "Na",
        "P",
        "S",
        "Zn",
    ]
x = df.drop(columns=target, errors='ignore')
x = x.drop(columns=['ID', 'Area', 'Depth_cm', 'ph'], errors='ignore')
y_columns = df.columns.difference(x.columns)
y_columns = [col for col in y_columns if col not in ["C_organic", "C_total", "Area", "Depth_cm", "ID"]]
y = df[y_columns]
y.fillna(0, inplace=True)
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [75]:
x_train.head()

,Longitude,Latitude,Cropland nitrogen per unit area,Cropland phosphorus per unit area,Cropland potassium per unit area,tmin_avg,tmax_avg,prec_avg,Bananas,"Beans, dry",...,Rice,"Seed cotton, unginned",Sesame seed,Sorghum,Soya beans,Sugar cane,Sweet potatoes,Tomatoes,Unmanufactured tobacco,Depth
4721,-2.19999,7.20972,-2.917073,-2.796745,-18.097209,22.215910,30.598484,103.590550,10390.800000,1165.118182,...,2719.254545,900.927273,0.000000,1169.454545,1634.036364,25018.527273,1781.009091,7377.254545,413.118182,0
7732,-11.01100,14.76583,3.931445,-2.550627,-3.758418,23.316288,37.174244,54.082947,34266.136364,811.044444,...,3123.272727,985.490909,420.227273,1037.981818,620.363636,72963.909091,17452.418182,17080.290909,2304.281818,0
5746,-1.07758,7.64100,-2.917073,-2.796745,-18.097209,22.660984,32.393940,103.886536,10390.800000,1165.118182,...,2719.254545,900.927273,0.000000,1169.454545,1634.036364,25018.527273,1781.009091,7377.254545,413.118182,0
12178,14.92311,-16.19789,9.219964,0.017264,-9.459173,15.484848,30.342804,61.349070,24064.809091,405.136364,...,1022.400000,1446.963636,263.136364,227.736364,603.709091,38876.818182,8256.936364,10544.109091,1018.445455,0
12952,15.22342,-12.29724,9.219964,0.017264,-9.459173,13.121212,25.611742,121.739590,24064.809091,405.136364,...,1022.400000,1446.963636,263.136364,227.736364,603.709091,38876.818182,8256.936364,10544.109091,1018.445455,0


In [76]:
y_train.head()

,Al,B,Ca,Cu,Fe,K,Mg,Mn,N,Na,P,S,Zn,ph
4721,668.664,0.0,2440.023,3.406,141.737,138.646,286.859,337.732,1.90,0.0,0.0,0.0,0.0,6.969
7732,981.937,0.0,2336.159,2.910,159.714,271.761,671.278,93.069,0.56,0.0,0.0,0.0,0.0,6.699
5746,837.742,0.0,2120.437,2.700,109.035,218.283,327.699,250.103,1.91,0.0,0.0,0.0,0.0,6.509
12178,730.772,0.0,740.481,0.966,172.359,87.640,134.619,102.362,0.80,0.0,0.0,0.0,0.0,6.161
12952,1145.199,0.0,541.405,2.794,83.008,160.257,149.018,69.130,0.62,0.0,0.0,0.0,0.0,5.903


In [77]:
models = {}
for col in y_columns:
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(x_train, y_train[col])
    y_pred = model.predict(x_test)
    rmse = root_mean_squared_error(y_test[col], y_pred)
    models[col] = model
    print(f"{col}: RMSE = {rmse}")

Al: RMSE = 130.2396468091458
B: RMSE = 0.0
Ca: RMSE = 959.4224269660466
Cu: RMSE = 0.7452205663210932
Fe: RMSE = 29.148475487969623
K: RMSE = 100.47139795477366
Mg: RMSE = 134.07210263822418
Mn: RMSE = 43.330061762577444
N: RMSE = 0.46218505583420955
Na: RMSE = 1.4646935033365307
P: RMSE = 0.0
S: RMSE = 0.0
Zn: RMSE = 0.0
ph: RMSE = 0.34927910272145685


In [78]:
for col, model in models.items():
    fi = pd.DataFrame(model.feature_importances_, index=x_train.columns, columns=['Feature Importance'])
    fi.sort_values(by='Feature Importance', ascending=False, inplace=True)
    print(f"Feature importance for {col}:")
    print(fi.head(10))

Feature importance for Al:
                                   Feature Importance
prec_avg                                     0.536625
Longitude                                    0.152047
Latitude                                     0.119135
tmin_avg                                     0.052261
Soya beans                                   0.046809
tmax_avg                                     0.022330
Pineapples                                   0.020283
Depth                                        0.012970
Coffee, green                                0.005833
Cropland phosphorus per unit area            0.004214
Feature importance for B:
                                                 Feature Importance
Longitude                                                       0.0
Seed cotton, unginned                                           0.0
Other fruits, n.e.c.                                            0.0
Other pulses n.e.c.                                             0.0
Other vegetab

# Generating test answers

## Importing df

In [12]:
train = read_data("../rhea-soil-nutrient-prediction-challenge/Train.csv")
train.head()

,ID,Longitude,Latitude,Depth_cm,ph,Area,Cropland nitrogen per unit area,Cropland phosphorus per unit area,Cropland potassium per unit area,tmin_avg,...,Cu,Fe,Mg,Mn,N,P,K,Na,S,Zn
0,BF9XTB,37.65189,-3.15440,20-50,6.405,Kenya,14.202173,3.902236,-7.263155,15.240530,...,5.826,81.780,306.836,270.240,0.79,NaN,300.951,NaN,NaN,NaN
1,2RWYTR,37.63612,-3.08585,20-50,6.419,Kenya,14.202173,3.902236,-7.263155,15.240530,...,4.346,97.198,407.980,185.557,1.11,NaN,292.696,NaN,NaN,NaN
2,XZI9Q6,39.55580,-2.67218,20-50,8.388,Kenya,14.202173,3.902236,-7.263155,21.969696,...,3.657,42.672,1256.319,178.299,0.45,NaN,814.911,NaN,NaN,NaN
3,4CBCVY,39.55477,-2.67196,20-50,8.302,Kenya,14.202173,3.902236,-7.263155,21.969696,...,3.376,52.861,1322.732,464.137,0.31,NaN,815.337,NaN,NaN,NaN
4,F9GK9S,39.55477,-2.67196,20-50,8.292,Kenya,14.202173,3.902236,-7.263155,21.969696,...,3.351,46.057,1134.898,274.565,0.45,NaN,928.238,NaN,NaN,NaN


In [13]:
test_df = pd.read_csv("../rhea-soil-nutrient-prediction-challenge/TestSet.csv")
test_df.head()

,ID,Latitude,Longitude,Depth_cm,Target_Al,Target_B,Target_Ca,Target_Cu,Target_Fe,Target_K,Target_Mg,Target_Mn,Target_N,Target_Na,Target_P,Target_S,Target_Zn
0,8ZMJRO,-0.746,37.094,0-20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,DCC6DM,-0.785,37.178,0-20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,T50LK1,-0.629,37.126,0-20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,FNLYT0,-0.351,35.308,0-20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,FP5E12,-1.894,36.987,0-20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [14]:
pred_to_keep = pd.read_csv("../rhea-soil-nutrient-prediction-challenge/TargetPred_To_Keep.csv")
pred_to_keep.head()

,ID,Al,B,Ca,Cu,Fe,K,Mg,Mn,N,Na,P,S,Zn
0,00A83S,1,1,1,1,1,1,1,1,1,1,1,1,1
1,00F2Q3,1,1,1,1,1,1,1,1,1,1,1,1,1
2,00FNFP,1,0,1,1,1,1,1,1,1,0,0,0,0
3,01MFSS,1,0,1,1,1,1,1,1,1,0,0,0,0
4,02851F,1,0,1,1,1,1,1,1,1,0,0,0,0


In [15]:
olek_features = pd.read_csv("../features/olek-features.csv")
print(olek_features.shape)
olek_features.head()

(50368, 9)


,ID,Country,Area,Cropland nitrogen per unit area,Cropland phosphorus per unit area,Cropland potassium per unit area,tmin_avg,tmax_avg,prec_avg
0,O2TONM,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.26878
1,BQLUK6,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.26878
2,LSET8M,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.26878
3,LEEL7I,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.727273,22.628788,123.26878
4,LDNGO2,Tanzania,Tanzania,6.777309,-2.492418,-7.380091,12.674242,23.077652,110.35551


In [16]:
mateusz_features = pd.read_csv("../features/mateusz-features.csv")
print(mateusz_features.shape)
mateusz_features.head()

(14, 27)


,Area,Bananas,"Beans, dry","Cashew nuts, in shell","Cassava, fresh","Chillies and peppers, green (Capsicum spp. and Pimenta spp.)","Coffee, green","Groundnuts, excluding shelled",Maize (corn),"Mangoes, guavas and mangosteens",...,Potatoes,Rice,"Seed cotton, unginned",Sesame seed,Sorghum,Soya beans,Sugar cane,Sweet potatoes,Tomatoes,Unmanufactured tobacco
0,Angola,24064.809091,405.136364,690.927273,11559.654545,0.000000,296.436364,560.190909,883.545455,11381.525000,...,7324.909091,1022.400000,1446.963636,263.136364,227.736364,603.709091,38876.818182,8256.936364,10544.109091,1018.445455
1,Botswana,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,446.145455,227.227273,0.000000,...,15454.050000,0.000000,2467.000000,0.000000,473.781818,0.000000,0.000000,11428.600000,44759.450000,0.000000
2,Burkina Faso,25250.216667,0.000000,892.290909,15187.918182,4513.727273,0.000000,799.118182,1679.145455,8623.972727,...,1926.981818,2197.463636,1193.318182,619.800000,982.563636,1300.981818,101775.372727,11356.209091,17100.609091,734.236364
3,Cameroon,15425.472727,1281.181818,0.000000,13845.209091,2126.245455,391.063636,1346.954545,1894.672727,5743.236364,...,9527.363636,1642.600000,1374.945455,919.781818,1467.390909,1257.536364,39950.227273,5891.345455,12658.981818,1503.327273
4,Côte d'Ivoire,43863.081818,855.745455,413.827273,6537.800000,6431.863636,169.409091,1273.509091,2054.745455,560.509091,...,0.000000,2448.181818,1035.763636,198.981818,679.290909,1305.281818,76121.318182,1904.236364,10221.927273,514.263636


## Combining data

In [17]:
features = pd.merge(olek_features, mateusz_features, on='Area', how = 'outer')
print(features.shape)
features.head()

(50368, 35)


,ID,Country,Area,Cropland nitrogen per unit area,Cropland phosphorus per unit area,Cropland potassium per unit area,tmin_avg,tmax_avg,prec_avg,Bananas,...,Potatoes,Rice,"Seed cotton, unginned",Sesame seed,Sorghum,Soya beans,Sugar cane,Sweet potatoes,Tomatoes,Unmanufactured tobacco
0,4DJZT0,Angola,Angola,9.219964,0.017264,-9.459173,15.409091,30.382576,59.48276,24064.809091,...,7324.909091,1022.4,1446.963636,263.136364,227.736364,603.709091,38876.818182,8256.936364,10544.109091,1018.445455
1,ZB5IOX,Angola,Angola,9.219964,0.017264,-9.459173,15.409091,30.382576,59.48276,24064.809091,...,7324.909091,1022.4,1446.963636,263.136364,227.736364,603.709091,38876.818182,8256.936364,10544.109091,1018.445455
2,ATCDI5,Angola,Angola,9.219964,0.017264,-9.459173,15.575758,30.361742,60.21079,24064.809091,...,7324.909091,1022.4,1446.963636,263.136364,227.736364,603.709091,38876.818182,8256.936364,10544.109091,1018.445455
3,9N85JO,Angola,Angola,9.219964,0.017264,-9.459173,15.409091,30.382576,59.48276,24064.809091,...,7324.909091,1022.4,1446.963636,263.136364,227.736364,603.709091,38876.818182,8256.936364,10544.109091,1018.445455
4,SMQQV9,Angola,Angola,9.219964,0.017264,-9.459173,15.575758,30.361742,60.21079,24064.809091,...,7324.909091,1022.4,1446.963636,263.136364,227.736364,603.709091,38876.818182,8256.936364,10544.109091,1018.445455


In [18]:
features.drop(columns=['Country', 'Area'], inplace=True)
features.head()

,ID,Cropland nitrogen per unit area,Cropland phosphorus per unit area,Cropland potassium per unit area,tmin_avg,tmax_avg,prec_avg,Bananas,"Beans, dry","Cashew nuts, in shell",...,Potatoes,Rice,"Seed cotton, unginned",Sesame seed,Sorghum,Soya beans,Sugar cane,Sweet potatoes,Tomatoes,Unmanufactured tobacco
0,4DJZT0,9.219964,0.017264,-9.459173,15.409091,30.382576,59.48276,24064.809091,405.136364,690.927273,...,7324.909091,1022.4,1446.963636,263.136364,227.736364,603.709091,38876.818182,8256.936364,10544.109091,1018.445455
1,ZB5IOX,9.219964,0.017264,-9.459173,15.409091,30.382576,59.48276,24064.809091,405.136364,690.927273,...,7324.909091,1022.4,1446.963636,263.136364,227.736364,603.709091,38876.818182,8256.936364,10544.109091,1018.445455
2,ATCDI5,9.219964,0.017264,-9.459173,15.575758,30.361742,60.21079,24064.809091,405.136364,690.927273,...,7324.909091,1022.4,1446.963636,263.136364,227.736364,603.709091,38876.818182,8256.936364,10544.109091,1018.445455
3,9N85JO,9.219964,0.017264,-9.459173,15.409091,30.382576,59.48276,24064.809091,405.136364,690.927273,...,7324.909091,1022.4,1446.963636,263.136364,227.736364,603.709091,38876.818182,8256.936364,10544.109091,1018.445455
4,SMQQV9,9.219964,0.017264,-9.459173,15.575758,30.361742,60.21079,24064.809091,405.136364,690.927273,...,7324.909091,1022.4,1446.963636,263.136364,227.736364,603.709091,38876.818182,8256.936364,10544.109091,1018.445455


In [19]:
test = pd.merge(test_df, features, on='ID', how='left')
test.head()

,ID,Latitude,Longitude,Depth_cm,Target_Al,Target_B,Target_Ca,Target_Cu,Target_Fe,Target_K,...,Potatoes,Rice,"Seed cotton, unginned",Sesame seed,Sorghum,Soya beans,Sugar cane,Sweet potatoes,Tomatoes,Unmanufactured tobacco
0,8ZMJRO,-0.746,37.094,0-20,0.0,0.0,0.0,0.0,0.0,0.0,...,15697.972727,3498.736364,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182
1,DCC6DM,-0.785,37.178,0-20,0.0,0.0,0.0,0.0,0.0,0.0,...,15697.972727,3498.736364,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182
2,T50LK1,-0.629,37.126,0-20,0.0,0.0,0.0,0.0,0.0,0.0,...,15697.972727,3498.736364,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182
3,FNLYT0,-0.351,35.308,0-20,0.0,0.0,0.0,0.0,0.0,0.0,...,15697.972727,3498.736364,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182
4,FP5E12,-1.894,36.987,0-20,0.0,0.0,0.0,0.0,0.0,0.0,...,15697.972727,3498.736364,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182


In [20]:
encoder = LabelEncoder()
train['Depth'] = encoder.fit_transform(train['Depth_cm'])
test['Depth'] = encoder.transform(test['Depth_cm'])
train.drop(columns=['ID', 'Area', 'Depth_cm'], inplace=True)
test.drop(columns=['ID', 'Depth_cm'], inplace=True)
test.head()

,Latitude,Longitude,Target_Al,Target_B,Target_Ca,Target_Cu,Target_Fe,Target_K,Target_Mg,Target_Mn,...,Rice,"Seed cotton, unginned",Sesame seed,Sorghum,Soya beans,Sugar cane,Sweet potatoes,Tomatoes,Unmanufactured tobacco,Depth
0,-0.746,37.094,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,3498.736364,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,0
1,-0.785,37.178,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,3498.736364,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,0
2,-0.629,37.126,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,3498.736364,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,0
3,-0.351,35.308,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,3498.736364,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,0
4,-1.894,36.987,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,3498.736364,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,0


In [21]:
train["Na"] = train["Na"].fillna(train["Na"].mean(), inplace=False)
train = train.fillna(0, inplace=False)
train.head()

,Longitude,Latitude,ph,Cropland nitrogen per unit area,Cropland phosphorus per unit area,Cropland potassium per unit area,tmin_avg,tmax_avg,prec_avg,Bananas,...,Fe,Mg,Mn,N,P,K,Na,S,Zn,Depth
0,37.65189,-3.15440,6.405,14.202173,3.902236,-7.263155,15.240530,25.774622,105.626890,21698.763636,...,81.780,306.836,270.240,0.79,0.0,300.951,32.149763,0.0,0.0,1
1,37.63612,-3.08585,6.419,14.202173,3.902236,-7.263155,15.240530,25.774622,105.626890,21698.763636,...,97.198,407.980,185.557,1.11,0.0,292.696,32.149763,0.0,0.0,1
2,39.55580,-2.67218,8.388,14.202173,3.902236,-7.263155,21.969696,30.452652,63.580307,21698.763636,...,42.672,1256.319,178.299,0.45,0.0,814.911,32.149763,0.0,0.0,1
3,39.55477,-2.67196,8.302,14.202173,3.902236,-7.263155,21.969696,30.452652,63.580307,21698.763636,...,52.861,1322.732,464.137,0.31,0.0,815.337,32.149763,0.0,0.0,1
4,39.55477,-2.67196,8.292,14.202173,3.902236,-7.263155,21.969696,30.452652,63.580307,21698.763636,...,46.057,1134.898,274.565,0.45,0.0,928.238,32.149763,0.0,0.0,1


## Separating Data

In [22]:
target = [
        "Al",
        "B",
        "Ca",
        "C_organic",
        "C_total",
        "Cu",
        "Fe",
        "K",
        "Mg",
        "Mn",
        "N",
        "Na",
        "P",
        "S",
        "Zn",
    ]
X = train.drop(columns=target, errors='ignore')
y_columns = train.columns.difference(X.columns)
y_columns = [col for col in y_columns if col not in ["C_organic", "C_total", "Area", "Depth_cm", "ID"]]
y = train[y_columns]


In [23]:
X.head()

,Longitude,Latitude,ph,Cropland nitrogen per unit area,Cropland phosphorus per unit area,Cropland potassium per unit area,tmin_avg,tmax_avg,prec_avg,Bananas,...,Rice,"Seed cotton, unginned",Sesame seed,Sorghum,Soya beans,Sugar cane,Sweet potatoes,Tomatoes,Unmanufactured tobacco,Depth
0,37.65189,-3.15440,6.405,14.202173,3.902236,-7.263155,15.240530,25.774622,105.626890,21698.763636,...,3498.736364,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,1
1,37.63612,-3.08585,6.419,14.202173,3.902236,-7.263155,15.240530,25.774622,105.626890,21698.763636,...,3498.736364,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,1
2,39.55580,-2.67218,8.388,14.202173,3.902236,-7.263155,21.969696,30.452652,63.580307,21698.763636,...,3498.736364,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,1
3,39.55477,-2.67196,8.302,14.202173,3.902236,-7.263155,21.969696,30.452652,63.580307,21698.763636,...,3498.736364,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,1
4,39.55477,-2.67196,8.292,14.202173,3.902236,-7.263155,21.969696,30.452652,63.580307,21698.763636,...,3498.736364,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,1


In [24]:
y.head()

,Al,B,Ca,Cu,Fe,K,Mg,Mn,N,Na,P,S,Zn
0,1108.121,0.0,1510.802,5.826,81.780,300.951,306.836,270.240,0.79,32.149763,0.0,0.0,0.0
1,1102.400,0.0,1592.621,4.346,97.198,292.696,407.980,185.557,1.11,32.149763,0.0,0.0,0.0
2,699.279,0.0,7541.051,3.657,42.672,814.911,1256.319,178.299,0.45,32.149763,0.0,0.0,0.0
3,905.183,0.0,5296.174,3.376,52.861,815.337,1322.732,464.137,0.31,32.149763,0.0,0.0,0.0
4,703.832,0.0,5922.011,3.351,46.057,928.238,1134.898,274.565,0.45,32.149763,0.0,0.0,0.0


In [25]:
test.head()

,Latitude,Longitude,Target_Al,Target_B,Target_Ca,Target_Cu,Target_Fe,Target_K,Target_Mg,Target_Mn,...,Rice,"Seed cotton, unginned",Sesame seed,Sorghum,Soya beans,Sugar cane,Sweet potatoes,Tomatoes,Unmanufactured tobacco,Depth
0,-0.746,37.094,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,3498.736364,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,0
1,-0.785,37.178,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,3498.736364,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,0
2,-0.629,37.126,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,3498.736364,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,0
3,-0.351,35.308,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,3498.736364,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,0
4,-1.894,36.987,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,3498.736364,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,0


In [26]:
target_pred = [
    "Target_Al",
    "Target_B",
    "Target_Ca",
    "Target_Cu",
    "Target_Fe",
    "Target_K",
    "Target_Mg",
    "Target_Mn",
    "Target_N",
    "Target_Na",
    "Target_P",
    "Target_S",
    "Target_Zn",
]

In [27]:
X_pred = test.drop(columns=target_pred, errors='ignore')
y_pred_columns = test.columns.difference(X_pred.columns)
y_pred_columns = [col for col in y_pred_columns if col not in ["C_organic", "C_total", "Area", "Depth_cm", "ID"]]
y_pred = test[y_pred_columns]

In [28]:
X_pred.head()

,Latitude,Longitude,Cropland nitrogen per unit area,Cropland phosphorus per unit area,Cropland potassium per unit area,tmin_avg,tmax_avg,prec_avg,Bananas,"Beans, dry",...,Rice,"Seed cotton, unginned",Sesame seed,Sorghum,Soya beans,Sugar cane,Sweet potatoes,Tomatoes,Unmanufactured tobacco,Depth
0,-0.746,37.094,14.202173,3.902236,-7.263155,14.543561,25.986742,108.980880,21698.763636,602.172727,...,3498.736364,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,0
1,-0.785,37.178,14.202173,3.902236,-7.263155,15.577652,27.297348,91.997140,21698.763636,602.172727,...,3498.736364,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,0
2,-0.629,37.126,14.202173,3.902236,-7.263155,14.102273,25.522728,115.549255,21698.763636,602.172727,...,3498.736364,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,0
3,-0.351,35.308,14.202173,3.902236,-7.263155,11.918561,24.070076,165.598710,21698.763636,602.172727,...,3498.736364,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,0
4,-1.894,36.987,14.202173,3.902236,-7.263155,14.017045,25.181818,45.469700,21698.763636,602.172727,...,3498.736364,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,0


In [29]:
y_pred.head()

,Target_Al,Target_B,Target_Ca,Target_Cu,Target_Fe,Target_K,Target_Mg,Target_Mn,Target_N,Target_Na,Target_P,Target_S,Target_Zn
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Prognosing ph feature

In [30]:
ph = X['ph']
X = X.drop(columns=['ph'], errors='ignore')
X.head()

,Longitude,Latitude,Cropland nitrogen per unit area,Cropland phosphorus per unit area,Cropland potassium per unit area,tmin_avg,tmax_avg,prec_avg,Bananas,"Beans, dry",...,Rice,"Seed cotton, unginned",Sesame seed,Sorghum,Soya beans,Sugar cane,Sweet potatoes,Tomatoes,Unmanufactured tobacco,Depth
0,37.65189,-3.15440,14.202173,3.902236,-7.263155,15.240530,25.774622,105.626890,21698.763636,602.172727,...,3498.736364,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,1
1,37.63612,-3.08585,14.202173,3.902236,-7.263155,15.240530,25.774622,105.626890,21698.763636,602.172727,...,3498.736364,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,1
2,39.55580,-2.67218,14.202173,3.902236,-7.263155,21.969696,30.452652,63.580307,21698.763636,602.172727,...,3498.736364,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,1
3,39.55477,-2.67196,14.202173,3.902236,-7.263155,21.969696,30.452652,63.580307,21698.763636,602.172727,...,3498.736364,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,1
4,39.55477,-2.67196,14.202173,3.902236,-7.263155,21.969696,30.452652,63.580307,21698.763636,602.172727,...,3498.736364,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,1


In [31]:
X_pred = X_pred[X.columns]
X_pred.head()

,Longitude,Latitude,Cropland nitrogen per unit area,Cropland phosphorus per unit area,Cropland potassium per unit area,tmin_avg,tmax_avg,prec_avg,Bananas,"Beans, dry",...,Rice,"Seed cotton, unginned",Sesame seed,Sorghum,Soya beans,Sugar cane,Sweet potatoes,Tomatoes,Unmanufactured tobacco,Depth
0,37.094,-0.746,14.202173,3.902236,-7.263155,14.543561,25.986742,108.980880,21698.763636,602.172727,...,3498.736364,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,0
1,37.178,-0.785,14.202173,3.902236,-7.263155,15.577652,27.297348,91.997140,21698.763636,602.172727,...,3498.736364,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,0
2,37.126,-0.629,14.202173,3.902236,-7.263155,14.102273,25.522728,115.549255,21698.763636,602.172727,...,3498.736364,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,0
3,35.308,-0.351,14.202173,3.902236,-7.263155,11.918561,24.070076,165.598710,21698.763636,602.172727,...,3498.736364,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,0
4,36.987,-1.894,14.202173,3.902236,-7.263155,14.017045,25.181818,45.469700,21698.763636,602.172727,...,3498.736364,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,0


In [32]:
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X, ph)
ph_pred = model.predict(X_pred)
ph_pred[:10]

array([6.13426   , 6.13722   , 6.19906   , 6.3341    , 7.10635333,
       6.3341    , 6.13722   , 6.25637   , 6.97199   , 7.13829333])

In [38]:
len(ph), len(ph_pred), X.shape, X_pred.shape

(13454, 6070, (13454, 35), (6070, 35))

In [41]:
X = pd.concat([X, ph], axis=1)
X.head()

,Longitude,Latitude,Cropland nitrogen per unit area,Cropland phosphorus per unit area,Cropland potassium per unit area,tmin_avg,tmax_avg,prec_avg,Bananas,"Beans, dry",...,"Seed cotton, unginned",Sesame seed,Sorghum,Soya beans,Sugar cane,Sweet potatoes,Tomatoes,Unmanufactured tobacco,Depth,ph
0,37.65189,-3.15440,14.202173,3.902236,-7.263155,15.240530,25.774622,105.626890,21698.763636,602.172727,...,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,1,6.405
1,37.63612,-3.08585,14.202173,3.902236,-7.263155,15.240530,25.774622,105.626890,21698.763636,602.172727,...,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,1,6.419
2,39.55580,-2.67218,14.202173,3.902236,-7.263155,21.969696,30.452652,63.580307,21698.763636,602.172727,...,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,1,8.388
3,39.55477,-2.67196,14.202173,3.902236,-7.263155,21.969696,30.452652,63.580307,21698.763636,602.172727,...,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,1,8.302
4,39.55477,-2.67196,14.202173,3.902236,-7.263155,21.969696,30.452652,63.580307,21698.763636,602.172727,...,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,1,8.292


In [42]:
X_pred['ph'] = ph_pred
X_pred.head()

,Longitude,Latitude,Cropland nitrogen per unit area,Cropland phosphorus per unit area,Cropland potassium per unit area,tmin_avg,tmax_avg,prec_avg,Bananas,"Beans, dry",...,"Seed cotton, unginned",Sesame seed,Sorghum,Soya beans,Sugar cane,Sweet potatoes,Tomatoes,Unmanufactured tobacco,Depth,ph
0,37.094,-0.746,14.202173,3.902236,-7.263155,14.543561,25.986742,108.980880,21698.763636,602.172727,...,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,0,6.134260
1,37.178,-0.785,14.202173,3.902236,-7.263155,15.577652,27.297348,91.997140,21698.763636,602.172727,...,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,0,6.137220
2,37.126,-0.629,14.202173,3.902236,-7.263155,14.102273,25.522728,115.549255,21698.763636,602.172727,...,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,0,6.199060
3,35.308,-0.351,14.202173,3.902236,-7.263155,11.918561,24.070076,165.598710,21698.763636,602.172727,...,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,0,6.334100
4,36.987,-1.894,14.202173,3.902236,-7.263155,14.017045,25.181818,45.469700,21698.763636,602.172727,...,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,0,7.106353


## Main model

In [43]:
models = {}
for idx, col in enumerate(y_pred_columns):
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X, y.iloc[:, idx])
    y_pred.iloc[:, idx] = model.predict(X_pred)
    models[col] = model
    print(col, y.iloc[:, idx])

Target_Al 0        1108.121
1        1102.400
2         699.279
3         905.183
4         703.832
           ...   
13449     349.000
13450    1650.000
13451     861.000
13452     170.000
13453     115.000
Name: Al, Length: 13454, dtype: float64
Target_B 0        0.0
1        0.0
2        0.0
3        0.0
4        0.0
        ... 
13449    0.0
13450    0.0
13451    0.0
13452    0.0
13453    0.0
Name: B, Length: 13454, dtype: float64
Target_Ca 0        1510.802
1        1592.621
2        7541.051
3        5296.174
4        5922.011
           ...   
13449     133.600
13450     174.000
13451     105.800
13452     352.000
13453     406.000
Name: Ca, Length: 13454, dtype: float64
Target_Cu 0        5.826
1        4.346
2        3.657
3        3.376
4        3.351
         ...  
13449    0.000
13450    0.000
13451    0.000
13452    0.000
13453    0.000
Name: Cu, Length: 13454, dtype: float64
Target_Fe 0        81.780
1        97.198
2        42.672
3        52.861
4        46.057
        

In [44]:
print(y_pred.shape)
y_pred.head()

(6070, 13)


,Target_Al,Target_B,Target_Ca,Target_Cu,Target_Fe,Target_K,Target_Mg,Target_Mn,Target_N,Target_Na,Target_P,Target_S,Target_Zn
0,1153.79982,0.0,1774.77435,3.07913,144.42270,313.34788,368.24652,149.69915,1.6101,32.149763,0.0,0.0,0.0
1,805.08088,0.0,1804.34296,3.02305,163.55377,308.45498,367.41338,135.61665,1.8048,32.149763,0.0,0.0,0.0
2,1101.15782,0.0,1753.55613,3.03721,139.80704,363.91775,370.67061,139.84850,1.6205,32.149763,0.0,0.0,0.0
3,1068.88681,0.0,2930.68942,3.64624,129.10182,419.27889,534.05336,163.26401,1.7709,32.149763,0.0,0.0,0.0
4,533.82699,0.0,3854.66128,3.69644,94.50107,716.52824,483.48550,149.17280,0.5231,32.149763,0.0,0.0,0.0


In [45]:
id = test_df["ID"]
result = pd.concat([id, y_pred], axis=1)
result.head()


,ID,Target_Al,Target_B,Target_Ca,Target_Cu,Target_Fe,Target_K,Target_Mg,Target_Mn,Target_N,Target_Na,Target_P,Target_S,Target_Zn
0,8ZMJRO,1153.79982,0.0,1774.77435,3.07913,144.42270,313.34788,368.24652,149.69915,1.6101,32.149763,0.0,0.0,0.0
1,DCC6DM,805.08088,0.0,1804.34296,3.02305,163.55377,308.45498,367.41338,135.61665,1.8048,32.149763,0.0,0.0,0.0
2,T50LK1,1101.15782,0.0,1753.55613,3.03721,139.80704,363.91775,370.67061,139.84850,1.6205,32.149763,0.0,0.0,0.0
3,FNLYT0,1068.88681,0.0,2930.68942,3.64624,129.10182,419.27889,534.05336,163.26401,1.7709,32.149763,0.0,0.0,0.0
4,FP5E12,533.82699,0.0,3854.66128,3.69644,94.50107,716.52824,483.48550,149.17280,0.5231,32.149763,0.0,0.0,0.0


In [46]:
pred_to_keep.shape

(6070, 14)

In [47]:
pred_to_keep.columns = result.columns
pred_to_keep.head()

,ID,Target_Al,Target_B,Target_Ca,Target_Cu,Target_Fe,Target_K,Target_Mg,Target_Mn,Target_N,Target_Na,Target_P,Target_S,Target_Zn
0,00A83S,1,1,1,1,1,1,1,1,1,1,1,1,1
1,00F2Q3,1,1,1,1,1,1,1,1,1,1,1,1,1
2,00FNFP,1,0,1,1,1,1,1,1,1,0,0,0,0
3,01MFSS,1,0,1,1,1,1,1,1,1,0,0,0,0
4,02851F,1,0,1,1,1,1,1,1,1,0,0,0,0


In [48]:
cols = pred_to_keep.columns.drop("ID")
result[cols] = result[cols].where(pred_to_keep[cols] == 1, other=0)
result.head()

,ID,Target_Al,Target_B,Target_Ca,Target_Cu,Target_Fe,Target_K,Target_Mg,Target_Mn,Target_N,Target_Na,Target_P,Target_S,Target_Zn
0,8ZMJRO,1153.79982,0.0,1774.77435,3.07913,144.42270,313.34788,368.24652,149.69915,1.6101,32.149763,0.0,0.0,0.0
1,DCC6DM,805.08088,0.0,1804.34296,3.02305,163.55377,308.45498,367.41338,135.61665,1.8048,32.149763,0.0,0.0,0.0
2,T50LK1,1101.15782,0.0,1753.55613,3.03721,139.80704,363.91775,370.67061,139.84850,1.6205,0.000000,0.0,0.0,0.0
3,FNLYT0,1068.88681,0.0,2930.68942,3.64624,129.10182,419.27889,534.05336,163.26401,1.7709,0.000000,0.0,0.0,0.0
4,FP5E12,533.82699,0.0,3854.66128,3.69644,94.50107,716.52824,483.48550,149.17280,0.5231,0.000000,0.0,0.0,0.0


In [49]:
result.to_csv("../rhea-soil-nutrient-prediction-challenge/submissions/ph-submission.csv", index=False)